<a href="https://colab.research.google.com/github/hristinagjorgjievska/Natural-Language-Processing/blob/master/lab2_vtornik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install evaluate
!pip install bert_score

In [2]:
import pandas as pd
from evaluate import load
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util

In [5]:
data = pd.read_csv('sarcasm.txt', sep='\t')

In [6]:
data

,Style 1,Style 2
0,The only thing I got from college is a caffein...,"College is really difficult, expensive, tiring..."
1,Remember the hundred emails from companies whe...,"I, at the bare minimum, wish companies actuall..."
2,Today my pop-pop told me I was not “forced” to...,"Today my pop-pop told me I was not ""forced"" to..."
3,@VolphanCarol @littlewhitty @mysticalmanatee I...,I would say Ted Cruz is an asshole and doesn’t...
4,Why would Alexa's recipe for Yorkshire pudding...,Great recipe from Alexa
...,...,...
688,"wow, ok, unfollowing now. Was a big fan of put...","I enjoy dancing the hokey cokey, in particular..."
689,Only posting selfie’s?? On YOUR own Instagram ...,It's my Instagram for a reason why wouldn't it...
690,do people with clear skin feel accomplished?? ...,people with clear skin are more comfortable in...
691,Update: holding hands with your mom and walkin...,Holding hands with your parent while walking a...


In [7]:
data = data.sample(frac=1, random_state=42, ignore_index=True)

In [11]:
train_end = int(0.8 * len(data))
val_end = int(0.1 * len(data))

In [12]:
train_data = data[:train_end]

In [13]:
val_data = data[train_end:train_end+val_end]

In [15]:
test_data = data[train_end+val_end:]

In [16]:
len(train_data), len(val_data), len(test_data)

(554, 69, 70)

===========================================================================

In [25]:
train_sarcasm = train_data['Style 1'].values.tolist()
train_sincere = train_data['Style 2'].values.tolist()

In [27]:
val_sarcasm = val_data['Style 1'].values.tolist()
val_sincere = val_data['Style 2'].values.tolist()

In [28]:
test_sarcasm = test_data['Style 1'].values.tolist()
test_sincere = test_data['Style 2'].values.tolist()

In [29]:
embedding_model = SentenceTransformer('all-distilroberta-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  328MB            

... (4 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
train_emb = embedding_model.encode(train_sarcasm, batch_size=64)

In [32]:
val_emb = embedding_model.encode(val_sarcasm, batch_size=64)

In [33]:
test_emb = embedding_model.encode(test_sarcasm, batch_size=64)

In [55]:
train_emb.shape, val_emb.shape, test_emb.shape

((554, 768), (69, 768), (70, 768))

===========================================================================

In [34]:
val_hits = util.semantic_search(val_emb, train_emb, top_k=5)

In [57]:
len(val_hits)

69

In [35]:
test_hits = util.semantic_search(test_emb, train_emb, top_k=5)

In [58]:
len(test_hits)

70

In [43]:
test_hits[0]

[{'corpus_id': 65, 'score': 0.3711138367652893},
 {'corpus_id': 84, 'score': 0.3710048198699951},
 {'corpus_id': 77, 'score': 0.3669975996017456},
 {'corpus_id': 227, 'score': 0.36540257930755615},
 {'corpus_id': 117, 'score': 0.3454122841358185}]

===========================================================================

In [59]:
bertscore = load('bertscore')

In [60]:
retrieved = [] #правиме празна листа, тука се собираат копираните преводи
for hits in test_hits: # ги земаме сите твит тестови еден по еден, за секој, hits се неговите 5 најсличн траин твита
  i = hits[0]['corpus_id'] # од тие 5, го земаме првиот [0]
  retrieved.append(train_sincere[i]) # го ставаме во листата

#retrieved е листа од 70 копирани преводи, по еден за секој тест твит

In [63]:
result = bertscore.compute(predictions=retrieved, references=test_sincere, model_type='distilroberta-base')

In [64]:
#bertscore.compute(...) - пресметај BERTScore
#predictions=retrieved - нашите одговори, 70-те копирани преводи
#references=test_sincere - точните одговори, вистинските 70 преводи
#model_type='distilroberta-base' - ко модел BERTScore го користи за да споредува значење
#споредува retrieved[0] со test_sincere[0], итн...

#result е речник со три листи, секоја со 70 броеви

In [65]:
retrieval_f1 = sum(result['f1']) / len(result['f1'])
retrieval_f1

0.8667723706790379